In [1]:
sample_count = 100
trial = 1

In [2]:
import re
import json
import pandas as pd
from bs4 import BeautifulSoup
from tqdm import tqdm
tqdm.pandas()

## Llama 7B

In [3]:
from langchain.chains import LLMChain

# llm
from langchain.callbacks.manager import CallbackManager
from langchain.callbacks.streaming_stdout import StreamingStdOutCallbackHandler
from langchain_community.llms import LlamaCpp

# Prompt
from langchain.chains.prompt_selector import ConditionalPromptSelector
from langchain.prompts import PromptTemplate

# Parser
from langchain_core.output_parsers import StrOutputParser

In [4]:
# TODO: Add better prompt that allows it to not force a boston location
# TODO: Make it give a formatted response so NER isn't necesssary.

# Prompt for LLM to do its geolocation task
prompt = PromptTemplate(
    input_variables=["headline", "body"],
    template="""<<SYS>> \n You are an assistant tasked in geo-locating \
this news article. \n <</SYS>> \n\n [INST] Generate a SHORT response \
of where you think this article is talking about. BE SPECIFIC AS POSSIBLE. IT IS IMPERATIVE THAT YOU HIGHLIGHT THE MOST SPECIFIC LOCATION. Give your response in the following format: \
1.Y/N indicating whether the article is talking about a region of Boston. \n 2.The specific location within the city you got if you got Y in the first question. \
3. The involved specific locations or organizations EXPLICITLY FOUND WITHIN THE ARTICLE that influenced your decision. \
If you do not know, PLEASE GIVE THE BEST GUESS AS POSSIBLE. \n\n
Headline: \n\n {headline} \n\n Body: \n\n {body} \n\n [/INST]""",
)

# prompt = PromptTemplate(
#     input_variables=["headline"],
#     template="""<<SYS>> \n You are an assistant tasked in geo-locating \
# this news article. \n <</SYS>> \n\n [INST] Generate a SHORT response \
# of where you think this article is talking about. BE SPECIFIC AND CONCISE AS POSSIBLE. IT IS IMPERATIVE THAT YOU HIGHLIGHT THE MOST SPECIFIC LOCATION. PLEASE CONSIDER THE CONTEXT OF THE ARTICLE. Give your response in the following format: \
# 1. A very brief summary of what the article is talking about. \n 2.The specific location you chose based on the context of the article. \
# If you do not know, PLEASE GIVE THE BEST GUESS AS POSSIBLE. \n\n
# Headline: \n\n {headline} \n\n [/INST]""",
# )

# prompt = PromptTemplate(
#     input_variables=["headline"],
#     template="""<<SYS>> \n You are an assistant tasked in geo-locating \
# this news article. \n <</SYS>> \n\n [INST] Generate a SHORT response \
# of where you think this article is talking about. BE SPECIFIC AND CONCISE AS POSSIBLE. IT IS IMPERATIVE THAT YOU HIGHLIGHT THE MOST SPECIFIC LOCATION. Give your response in the following format: \
# 1.Y/N indicating whether the article is talking about a region of Boston \n 2.The specific location within the city you got if you got Y in the first question. \
# If you do not know, PLEASE GIVE THE BEST GUESS AS POSSIBLE. PLEASE KEEP YOUR ANSWER SHORT. \n\n
# Headline: \n\n {headline} \n\n Body: \n\n {body}  \n\n [/INST]""",
# )

In [5]:
# Call model. Needs to be downloaded
llama_model_path = "./models/llama_7B/llama-2-7b-chat.Q4_K_M.gguf"
llama2_model_path = "./models/llama_7B/llama-2-7b-chat.Q4_K_M.gguf"
llama3_1_model_path = "./models/llama_3_1_8B/Meta-Llama-3.1-8B-Instruct-Q4_K_M.gguf"


In [6]:
# llm = LlamaCpp(
#     model_path=llama_model_path,
#     n_gpu_layers=1,
#     n_batch=1024,
#     n_ctx=2048,
#     f16_kv=True,
#     callback_manager=CallbackManager([StreamingStdOutCallbackHandler()]),
#     verbose=True,
# )
# output_parser = StrOutputParser()

In [7]:
llm2 = LlamaCpp(
    model_path=llama2_model_path,
    n_gpu_layers=1,
    n_batch=1024,
    n_ctx=2048,
    f16_kv=True,
)
output_parser = StrOutputParser()

llama_model_loader: loaded meta data with 19 key-value pairs and 291 tensors from ./models/llama_7B/llama-2-7b-chat.Q4_K_M.gguf (version GGUF V2)
llama_model_loader: Dumping metadata keys/values. Note: KV overrides do not apply in this output.
llama_model_loader: - kv   0:                       general.architecture str              = llama
llama_model_loader: - kv   1:                               general.name str              = LLaMA v2
llama_model_loader: - kv   2:                       llama.context_length u32              = 4096
llama_model_loader: - kv   3:                     llama.embedding_length u32              = 4096
llama_model_loader: - kv   4:                          llama.block_count u32              = 32
llama_model_loader: - kv   5:                  llama.feed_forward_length u32              = 11008
llama_model_loader: - kv   6:                 llama.rope.dimension_count u32              = 128
llama_model_loader: - kv   7:                 llama.attention.head_count u

In [8]:
prompt3_1 = PromptTemplate(
    input_variables=["headline", "body"],
    template="""
    <|begin_of_text|><|start_header_id|>system<|end_header_id|>
    
    Cutting Knowledge Date: December 2023
    Today Date: 26 Jul 2024

    You are an expert in geo-location and have a deep understanding of specific places and organizations. In a short response, your task is to identify and provide the most exact real location mentioned in the news article. This can be a place, organization, facility, or any location that can help identify where the article takes place or talks about. Also, mention any specific locations or organizations explicitly found within the article that influenced your decision. If you cannot determine a location, state that explicitly. Do not discuss anthing else.<|eot_id|><|start_header_id|>user<|end_header_id|>

    You are tasked in geo-locating this news article. Generate a SHORT response specifying the most exact location you can find mentioned in the article.  Highlight the most specific location, including the city, and mention any specific locations or organizations explicitly found within the article that influenced your decision. If you cannot determine a location, state that explicitly. DO NOT MAKE UP INFORMATION.
    Also, give the involved specific locations or organizations EXPLICITLY FOUND WITHIN THE ARTICLE that influenced your decision. This is the news article:
    Headline: \n\n {headline} \n\n Body: \n\n {body} \n\n <|eot_id|><|start_header_id|>assistant<|end_header_id|>""",
)

In [9]:
llm3_1 = LlamaCpp(
    model_path=llama3_1_model_path,
    n_gpu_layers=1,
    n_batch=1024,
    n_ctx=2048,
    f16_kv=True,
)
output_parser = StrOutputParser()

llama_model_loader: loaded meta data with 33 key-value pairs and 292 tensors from ./models/llama_3_1_8B/Meta-Llama-3.1-8B-Instruct-Q4_K_M.gguf (version GGUF V3 (latest))
llama_model_loader: Dumping metadata keys/values. Note: KV overrides do not apply in this output.
llama_model_loader: - kv   0:                       general.architecture str              = llama
llama_model_loader: - kv   1:                               general.type str              = model
llama_model_loader: - kv   2:                               general.name str              = Meta Llama 3.1 8B Instruct
llama_model_loader: - kv   3:                           general.finetune str              = Instruct
llama_model_loader: - kv   4:                           general.basename str              = Meta-Llama-3.1
llama_model_loader: - kv   5:                         general.size_label str              = 8B
llama_model_loader: - kv   6:                            general.license str              = llama3.1
llama_model_l

In [10]:
chain2 = prompt | llm2 | output_parser
chain3_1 = prompt3_1 | llm3_1 | output_parser

In [11]:
# Run LLM on a given article
def run_llm2(headline, body):
    return chain2.invoke({"headline": headline, "body": body})

def run_llm3_1(headline, body):
    return chain3_1.invoke({"headline": headline, "body": body})


## NER Model

In [12]:
import spacy
from span_marker import SpanMarkerModel

In [13]:
# Load the spacy model with the span_marker pipeline component
nlp = spacy.load("en_core_web_sm", exclude=["ner"])
nlp.add_pipe("span_marker", config={"model": "tomaarsen/span-marker-roberta-large-ontonotes5"})

## Google Maps

In [14]:
# Load environment variables
import os
from dotenv import load_dotenv

load_dotenv()
gmap_api_key =  os.getenv('GMAP_API_KEY')

In [15]:
import ast
import requests
import googlemaps
from mapbox import Geocoder

In [16]:
gmap_client_key = gmap_api_key
gmaps = googlemaps.Client(key=gmap_client_key)

In [17]:
# Google Maps API handler
def callGoogleMapsAPI(location):
    try:
        # If it's in boston, we can do a more specific search
        if ("Boston" in location):
            location = f"{location}, Boston"
            
        # Locations are limited to Massachusetts for now
        geocode_result = gmaps.geocode(f"{location}, Massachussetts", components={"administrative_area_level": "MA", "country": "US"})
        
        if (len(geocode_result) > 0):
            longitude = geocode_result[0]['geometry']['location']['lng']
            latitude = geocode_result[0]['geometry']['location']['lat']
            return longitude, latitude
        else:
            return None
    except Exception as error:
        print(error)
        return None

### Functions to manage caches

In [18]:
# Load the cache from the file at the start
def load_cache(path):
    try:
        with open(path, 'r') as file:
            cache = json.load(file)
    except FileNotFoundError:
        cache = {}
    return cache

# Save cache to file
def save_cache_to_file(cache, path):
    with open(path, 'w') as file:
        json.dump(cache, file, indent=4)

### Wrapper function to measure time taken by a given function

In [19]:
import time

def sec_to_hms(seconds):
    hours = int(seconds // 3600)
    minutes = int((seconds % 3600) // 60)
    remaining_seconds = round(seconds % 60)
    return f"{hours:02}:{minutes:02}:{remaining_seconds:02}"

def check_time(func):
    
    def wrapper(*args, **kwargs):
        start_time = time.time()
        result = func(*args, **kwargs)
        end_time = time.time()
        total_time = end_time - start_time
        total_time_formatted = sec_to_hms(total_time)
        print(f"Time taken: {total_time_formatted}")
        return result
    return wrapper

## Pipeline Entry Point

In [20]:
sample_data_path = "./sample_data/Articles_Nov_2020_March_2023.csv" # Using this as I don't have the other one
# sample_data_dir = "./sample_data/se_naacp_db.articles_data.csv"

In [21]:
# Temporary. Use given article data set. Comment out when obtain the other data sate
full_df = pd.read_csv(sample_data_path)

# Format data set to match expected pipeline input
full_df = full_df.rename(columns={"Headline": "hl1", "Body": "body"})

# Make 'tagging' column be the id column
tagging_col = full_df.pop('Tagging')
full_df.insert(0, '_id', tagging_col)

# Drop rows where at least one of the specified columns is empty
columns_to_check = ['_id', 'hl1', 'body'] 
full_df = full_df.dropna(subset=columns_to_check, how='all')

# Drop empty rows too
full_df = full_df[~full_df['body'].apply(lambda x: isinstance(x, float))]
full_df = full_df[~full_df['hl1'].apply(lambda x: isinstance(x, float))]

# Pick a random sample of 5 articles
raw_df = full_df.sample(sample_count)
# raw_df = full_df
len(raw_df)


100

In [22]:
# raw_df = pd.read_csv(sample_data_path)

In [23]:
raw_df.head(10)

,_id,Type,Label,hl1,Byline,Section Navigation,Section,Title,Paths,Publish Date,Has Path?,body
7702,0000017f-263d-d19f-a3ff-ef3d8a100001,Article,Florida has become the GOP's favorite destinat...,Florida has become the GOP's favorite destinat...,Greg Allen,NaN,National News,NaN,/national-news/2022/02/23/florida-has-become-t...,Wed Feb 23 05:00:00 EST 2022,TRUE,Florida is a big draw for snow birds from arou...
2936,00000178-56fd-da59-a37b-d7fd12e40001,Article,"A Year Into Pandemic, Veterans Halls 'Barely H...","A Year Into Pandemic, Veterans Halls 'Barely H...",Philip Marcelo | Associated Press,NaN,Local News,NaN,/local-news/2021/03/21/a-year-into-pandemic-ve...,Sun Mar 21 18:57:54 EDT 2021,TRUE,"NEW BEDFORD, Mass. (AP) — Paul Guilbeault knew..."
6426,0000017d-203f-d269-a3fd-be3ffda40000,Article,"3 dead, hundreds injured after storms rouse sc...","3 dead, hundreds injured after storms rouse sc...",Sharon Pruitt-Young,NaN,International News,NaN,/international-news/2021/11/15/3-dead-hundreds...,Sun Nov 14 15:08:00 EST 2021,TRUE,Three people are dead and hundreds are injured...
6693,0000017d-9619-d0ab-a17d-deffaa9e0001,Article,"Boston Ballet's ""The Gift""","Boston Ballet's ""The Gift""",GBH Music Staff,NaN,NaN,NaN,/music/boston-ballet-the-gift (Permalink),Wed Dec 08 11:04:45 EST 2021,TRUE,Enjoy a special preview of Boston Ballet&#39;s...
4389,0000017a-26b7-dc5c-a77a-2ef7a0a40001,Article,"Driver Rams Cyclists In Arizona Race, Critical...","Driver Rams Cyclists In Arizona Race, Critical...",The Associated Press,NaN,National News,NaN,/national-news/2021/06/20/driver-rams-cyclists...,Sat Jun 19 19:50:00 EDT 2021,TRUE,"SHOW LOW, Ariz. (AP) — A driver in a pickup tr..."
2581,00000177-f7bf-d081-ad7f-ffbf11190001,Article,Drug Overdose Deaths Surge Among Black America...,Drug Overdose Deaths Surge Among Black America...,Brian Mann,NaN,National News,NaN,/national-news/2021/03/03/drug-overdose-deaths...,Wed Mar 03 05:00:00 EST 2021,TRUE,"When Latoya Jenkins talks about her mom, she l..."
2344,00000177-ba85-d244-a57f-fbfd3e720001,Article,'The Outlaw And The Lawman': A Remembrance Of ...,'The Outlaw And The Lawman': A Remembrance Of ...,00000177-ba85-d244-a57f-fbfd3e720000,NaN,Local News,NaN,/local-news/2021/02/19/the-outlaw-and-the-lawm...,Fri Feb 19 09:39:23 EST 2021,TRUE,It was the classic tale of the outlaw and the ...
1776,00000177-2ad9-d79b-abff-abffb43c0001,Article,LISTEN: American Democracy — Buy Or Sell?,LISTEN: American Democracy — Buy Or Sell?,Adam Reilly,NaN,Politics,NaN,/politics/2021/01/22/listen-american-democracy...,Fri Jan 22 11:25:45 EST 2021,TRUE,After the 2020 presidential election and the h...
10526,00000183-af56-d227-a9b7-bfd7c9ce0001,Article,Boston's Latino business owners still lacking ...,Boston's Latino business owners still lacking ...,Alexi Cohan,NaN,Local News,NaN,/local-news/2022/10/07/bostons-latino-business...,Fri Oct 07 09:24:11 EDT 2022,TRUE,Dr. Rosa Calcaño of Boston owns two businesses...
2748,00000178-235f-da59-a37b-a7ffd5f40001,Article,Campbell And Wu Criticize Walsh's Handling Of ...,Campbell And Wu Criticize Walsh's Handling Of ...,Saraya Wintersmith,NaN,Politics,NaN,/politics/2021/03/11/campbell-and-wu-criticize...,Thu Mar 11 20:41:58 EST 2021,TRUE,Two candidates to replace Mayor Marty Walsh ha...


The ML Model honestly just needs the `id`, `header`, and `body`.

In [24]:
df = pd.concat([raw_df['_id'], raw_df['hl1'], raw_df['body']], axis=1)

In [25]:
# For Testing Purposes Only
# df = df[:20]

Remove Duplicates (if any)

In [26]:
duplicates = df.duplicated(subset=['hl1'])

In [27]:
print(duplicates.value_counts())

False    100
Name: count, dtype: int64


In [28]:
df = df.drop_duplicates(subset=['hl1'])

Clean the HTML in the body and header

In [29]:
# Function to extract the text from the html of the article
func_clean_html = lambda text: BeautifulSoup(text, "html.parser").get_text()
df['body'] = df['body'].progress_apply(func_clean_html)
df['hl1'] = df['hl1'].progress_apply(func_clean_html)

100%|██████████| 100/100 [00:00<?, ?it/s]


Clean the Body and Header with Regex

In [30]:
# Function to remove extra symbols from the text
func_clean_regex = lambda text: ' '.join([word for word in re.findall(r'[A-Za-z0-9!@#$%^&*().]+', text) if len(word) > 1])
df['body'] = df['body'].progress_apply(func_clean_regex)
df['hl1'] = df['hl1'].progress_apply(func_clean_regex)

100%|██████████| 100/100 [00:00<00:00, 214542.40it/s]


### Benchmark total times

In [31]:
time_df = pd.DataFrame(columns=['NER', 'LLM2', 'LLM3.1'])


### Explicit Article Mentions

Load the well-known locations, organizations, and neighborhoods dictionary and the locations that we don't want to allow (i.e. Too broad or incorrect ones like "Boston", "Massachussets", etc.)

In [32]:
known_title_locs_path = "./geodata/known_locs.json"
known_title_locs = load_cache(known_title_locs_path)

unwanted_entities_path = "./geodata/unwanted_locations.json"  
unwanted_entities = load_cache(unwanted_entities_path)

In [33]:
# If a location is in the title, use that as the article's location
def explicit_filtering(header):
    # Look through the header for known locations
    lowercase_header = header.lower()
    for location in known_title_locs.keys():
        if (location.lower() in lowercase_header):
            if location not in unwanted_entities["FAC"]:
                return location
    return None

In [34]:
# df["Explicit_Pass"] = df["hl1"].progress_apply(explicit_filtering)
df["Explicit_Pass"] = None


In [35]:
df["Explicit_Pass"].value_counts().head(10)

Series([], Name: count, dtype: int64)

### NER Code First Pass

In [36]:
# Return the first valid facility found, or organization if none are found
def valid_facility(entities, firstPass):
    print(entities)

    if (firstPass): 
        for entity in entities:
            # If it's a valid facility, return it
            if (entity.label_ == "FAC" and entity.text not in unwanted_entities["FAC"]):
                return entity.text
        else:
            return None
    
    # Process for the LLM Prediction Pass
    else:
        first_org = None
        for entity in entities:
            # If it's a valid facility, return it
            if (entity.label_ == "FAC" and entity.text not in unwanted_entities["FAC"]):
                return entity.text
            
            # If it's a valid organization, save it (but don't return in case there's a facility later on)
            if (first_org == None and entity.label_ == "ORG" and entity.text not in unwanted_entities["ORG"]):
                first_org = entity.text
        else:             
            return first_org # Return regardless of whether it's None or not 
        

In [37]:
# Run NER on the body of the article and return first valid facility
def run_NER(text, firstPass=True):
    try:
        if (text == None or text == ""):
            return None
        
        entities = nlp(text).ents
        return valid_facility(entities, firstPass)
        
    except Exception as error:
        print(error)
        return None

In [38]:
# Chunk processing - split the article into chunks of text and run NER on each chunk
chunk_size = 100
def chunk_processing(text, chunk_size=chunk_size):
    # Split text into smaller chunks
    chunks = split_text_into_chunks(text, chunk_size)

    # Process each chunk and return if a valid facilty is found
    for chunk in chunks:
        result = run_NER(chunk)
        if result is not None:
             return result
    return None

# Split article text into chunks of specified size
def split_text_into_chunks(text, chunk_size=chunk_size):
    words = text.split()
    chunks = [' '.join(words[i:i + chunk_size]) for i in range(0, len(words), chunk_size)]
    return chunks

In [39]:
@check_time
def handle_chunk_processing(article):
    # Filter out articles that have no text
    text = article['body']
    if (text is None or text == ""):
            return None
    try:
        return chunk_processing(text)
    except Exception as error:
        print(error)

In [40]:
start_time = time.time()

df["NER_Pass"] = df.progress_apply(handle_chunk_processing, axis=1)
# df["NER_Pass"] = None

end_time = time.time()
total_time = end_time - start_time
total_time_formatted = sec_to_hms(total_time)
print(f"Time taken: {total_time_formatted}")
time_df.loc[0, "NER"] = total_time_formatted


  0%|          | 0/100 [00:00<?, ?it/s]

(Florida, winter, Republicans, year round, Donald Trump, Wednesday, Palm Beach, Thursday, CPAC, the Conservative Political Action Conference, Orlando, annual, Washington D.C., last year, Florida, House, GOP, the Sunshine State, this year)
(Republican, first, COVID 19, CPAC, Matt Schlapp, last year, Maryland, Schlapp, Florida, Republican, Ron DeSantis, GOP, Biden, DeSantis)
(DeSantis, CPAC, several decades, the last decade, Maryland, Schlapp, Florida)


  2%|▏         | 2/100 [01:08<56:18, 34.47s/it]

(Maryland, Republican, Florida, Federalist Society, Mike Pence, Trump, The Republican National Committee, spring, last year, Mar Lago, Trump)
Time taken: 00:01:09
(NEW BEDFORD, Mass., AP, Paul Guilbeault, Veterans of Foreign Wars, Boston, Massachusetts, last March, six months, 90 year old, Korean War, VFW Post 3260, New Bedford, 1935, Guilbeault, years)
(VFW, American Legion, years, COVID 19, The U.S. Department of Veterans Affairs)
(last spring, Lakeview, Michigan, VFW Post 3701, hundreds, the Red Cross, Queens, New York, American Legion Post 483, thousands, Connecticut, North)
(Carolina, Harold Durr, American Legion Post, Santa Fe, New Mexico, Durr)
(monthly, 75 year old, Navy, the Vietnam War, 100 years, VFW, American Legion)
(year, prior years, last fall, thousands of dollars, hundreds, John Raughter, Indianapolis, American Legion)
(Scout, the VFW Post 2718, New York, first, Mother Day, John McManamy, Massachusetts, New Bedford, some 20%, VFW)
(summer months, Bill LeBeau, VFW, Mass

  3%|▎         | 3/100 [04:13<2:36:42, 96.93s/it]

...

(Today, Rath, today, Feldman, the United States, Capitol)
Time taken: 00:01:08


100%|██████████| 100/100 [2:11:18<00:00, 37.44s/it]

(This weekend, Wicked Queer Boston LGBTQ Film Festival, first, 1984, Nov. 21, the Museum of Fine Arts, the Brattle Theatre, NELLY NADINE, World War II, Casa Susanna, Catskills, 50s, 60s, Shawn Cotter, Wicked Queer, GBH, All Things Considered, Arun Rath)
Time taken: 00:00:16
(WASHINGTON, AP, Joe Biden, one, Biden, American, Tuesday)
(end of May, two months, Biden, full year, the end of May, Biden, April)
(Americans, Biden, Americans, his first days, Biden, the end of summer, the end of July, Pfizer, Moderna, This past weekend, Johnson Johnson, Merck, Biden, Tuesday)
(May, Biden, Tuesday, this time next year, Biden)
(100 million, his first 100 days, Christmas, Tuesday)
(Donald Trump, Biden, Thursday, the Centers for Disease Control)
(CDC, Tom Frieden, January, one, Frieden, Resolve to Save Lives)
(COVID 19, Frieden, the next several weeks, fourth, Trump, Americans)
(last Easter, nearly 11 months ago, weeks, Trump, fall, first)
(Alex Conant, Marco Rubio, 2016, Trump, Biden, Tuesday, Biden

100%|██████████| 100/100 [2:14:16<00:00, 80.56s/it]

(Conant, Americans, Lemire, New York)
Time taken: 00:02:57
Time taken: 02:14:16


In [41]:
df.head(10)

,_id,hl1,body,Explicit_Pass,NER_Pass
7702,0000017f-263d-d19f-a3ff-ef3d8a100001,Florida has become the GOP favorite destinatio...,Florida is big draw for snow birds from around...,None,Mar Lago
2936,00000178-56fd-da59-a37b-d7fd12e40001,Year Into Pandemic Veterans Halls Barely Hangi...,NEW BEDFORD Mass. (AP) Paul Guilbeault knew th...,None,None
6426,0000017d-203f-d269-a3fd-be3ffda40000,dead hundreds injured after storms rouse scorp...,Three people are dead and hundreds are injured...,None,None
6693,0000017d-9619-d0ab-a17d-deffaa9e0001,Boston Ballet The Gift,Enjoy special preview of Boston Ballet The Gif...,None,None
4389,0000017a-26b7-dc5c-a77a-2ef7a0a40001,Driver Rams Cyclists In Arizona Race Criticall...,SHOW LOW Ariz. (AP) driver in pickup truck plo...,None,U.S. 60
2581,00000177-f7bf-d081-ad7f-ffbf11190001,Drug Overdose Deaths Surge Among Black America...,When Latoya Jenkins talks about her mom she li...,None,None
2344,00000177-ba85-d244-a57f-fbfd3e720001,The Outlaw And The Lawman Remembrance Of Class...,It was the classic tale of the outlaw and the ...,None,None
1776,00000177-2ad9-d79b-abff-abffb43c0001,LISTEN American Democracy Buy Or Sell,After the 2020 presidential election and the h...,None,None
10526,00000183-af56-d227-a9b7-bfd7c9ce0001,Boston Latino business owners still lacking ac...,Dr. Rosa Calca of Boston owns two businesses b...,None,Seaport
2748,00000178-235f-da59-a37b-a7ffd5f40001,Campbell And Wu Criticize Walsh Handling Of Su...,Two candidates to replace Mayor Marty Walsh ha...,None,None


### Llama Prediction

In [42]:
# Run LLM model on the articles, and then run the NER on the prediction.
@check_time
def predict_llama2(article):
    try:
        truncated_text = article['body'][:6000]
        llama_prediction = run_llm2(article['hl1'], truncated_text)
        print(llama_prediction)
        return run_NER(llama_prediction, False)
    except Exception as error:
        print(error)
        return None

In [43]:
start_time = time.time()

df['LLM_2_Pass'] = df.progress_apply(predict_llama2, axis=1)
# df['LLM_2_Pass'] = None

end_time = time.time()
total_time = end_time - start_time
total_time_formatted = sec_to_hms(total_time)
print(f"Total time taken: {total_time_formatted}")
time_df.loc[0, "LLM2"] = total_time_formatted

  0%|          | 0/100 [00:00<?, ?it/s]
llama_print_timings:        load time =   85705.68 ms
llama_print_timings:      sample time =      84.46 ms /   256 runs   (    0.33 ms per token,  3031.09 tokens per second)
llama_print_timings: prompt eval time =  104507.23 ms /  1233 tokens (   84.76 ms per token,    11.80 tokens per second)
llama_print_timings:        eval time =   45973.06 ms /   255 runs   (  180.29 ms per token,     5.55 tokens per second)
llama_print_timings:       total time =  150985.89 ms /  1488 tokens


  1. Y - The article is talking about a region of Florida, specifically Palm Beach and Orlando.
2. Specific location within the city: Palm Beach and Orlando are the specific locations mentioned in the article as the venues for various Republican events, including the CPAC gathering and fundraisers at Mar-a-Lago.
3. Involved specific locations or organizations explicitly found within the article that influenced my decision:
* Palm Beach - The article mentions former President Donald Trump hosting congressional candidates at a big fundraiser at his private club in Palm Beach.
* Orlando - CPAC, the Conservative Political Action Conference, is held annually in Orlando, and this year's gathering was moved from Washington D.C. area until last year when it moved to Florida.
* Mar-a-Lago - The article highlights Mar-a-Lago as a popular destination for Republican fundraisers, with numerous events held at the club last year, including fundraisers for at least 20 Republican candidates.
The above 

  2%|▏         | 2/100 [03:12<2:37:02, 96.15s/it]

(1, Florida, Palm Beach, Orlando, 2, Palm Beach, Orlando, Republican, CPAC, Mar-a-Lago, 3, Palm Beach, Donald Trump, Palm Beach, Orlando - CPAC, the Conservative Political Action Conference, annually, Orlando, year, Washington D.C., last year, Florida, Mar-a-Lago, Mar-a-Lago, Republican, last year, at least 20, Republican, Palm Beach, Orlando, Florida, Republican)
Time taken: 00:03:12


Llama.generate: prefix-match hit

llama_print_timings:        load time =   85705.68 ms
llama_print_timings:      sample time =      80.29 ms /   256 runs   (    0.31 ms per token,  3188.52 tokens per second)
llama_print_timings: prompt eval time =  114182.69 ms /  1364 tokens (   83.71 ms per token,    11.95 tokens per second)
llama_print_timings:        eval time =   46944.94 ms /   255 runs   (  184.10 ms per token,     5.43 tokens per second)
llama_print_timings:       total time =  161555.38 ms /  1619 tokens


...

(Boston, Massachusetts, the Museum of Fine Arts, the Brattle Theatre, 1, The Museum of Fine Arts, one, 2, The Brattle Theatre, 3, Casa Susanna, the 50s and 60s, one, 4, Nellie Nadine, World War II, one)
Time taken: 00:02:40


Llama.generate: prefix-match hit

llama_print_timings:        load time =   85705.68 ms
llama_print_timings:      sample time =      82.09 ms /   239 runs   (    0.34 ms per token,  2911.51 tokens per second)
llama_print_timings: prompt eval time =  108731.04 ms /  1352 tokens (   80.42 ms per token,    12.43 tokens per second)
llama_print_timings:        eval time =   43294.76 ms /   238 runs   (  181.91 ms per token,     5.50 tokens per second)
llama_print_timings:       total time =  152413.84 ms /  1590 tokens


  Based on the information provided in the article, I would guess that the location being referred to is most likely Boston, Massachusetts. The article mentions President Joe Biden's pledge to have enough vaccine for all adults by the end of May, which suggests that the location is within the United States. Additionally, the article cites a specific date of April 15th as the point at which administration officials expect the supply of vaccine to outpace demand, which implies that the location is likely in the eastern time zone.
As for the specific locations or organizations involved in the article, it mentions:
* The Biden administration
* The Centers for Disease Control and Prevention (CDC)
* Public health experts, including former CDC Director Tom Frieden
* Sen. Marco Rubio's 2016 presidential campaign

Based on these references, it seems that the article is focused on the efforts of the Biden administration to manage expectations and coordinate vaccine distribution in response to th

100%|██████████| 100/100 [3:34:58<00:00, 128.99s/it]

(Boston, Massachusetts, Joe Biden, the end of May, the United States, April 15th, Biden, The Centers for Disease Control and Prevention, CDC, CDC, Tom Frieden, Marco Rubio, 2016, Biden, COVID-19)
Time taken: 00:03:03
Total time taken: 03:34:59


In [44]:
@check_time
def predict_llama3_1(article):
    try:
        truncated_text = article['body'][:6000]
        llama_prediction = run_llm3_1(article['hl1'], truncated_text)
        print(llama_prediction)
        return run_NER(llama_prediction, False)
    except Exception as error:
        print(error)
        return None

In [45]:
start_time = time.time()

df['LLM_3_1_Pass'] = df.progress_apply(predict_llama3_1, axis=1)

end_time = time.time()
total_time = end_time - start_time
total_time_formatted = sec_to_hms(total_time)
print(f"Total time taken: {total_time_formatted}")
time_df.loc[0, "LLM3.1"] = total_time_formatted

  0%|          | 0/100 [00:00<?, ?it/s]
llama_print_timings:        load time =   96255.41 ms
llama_print_timings:      sample time =     111.54 ms /   107 runs   (    1.04 ms per token,   959.31 tokens per second)
llama_print_timings: prompt eval time =  105895.79 ms /  1136 tokens (   93.22 ms per token,    10.73 tokens per second)
llama_print_timings:        eval time =   19362.72 ms /   106 runs   (  182.67 ms per token,     5.47 tokens per second)
llama_print_timings:       total time =  125762.68 ms /  1242 tokens




The most exact real location mentioned in the news article is:

Palm Beach, Florida.

Specific locations or organizations explicitly found within the article that influenced my decision include:

1. Mar-a-Lago (private club owned by Donald Trump)
2. CPAC (Conservative Political Action Conference) at the Hyatt Regency hotel in Orlando, Florida.
3. The Republican National Committee (RNC) events and meetings held in Florida.

These locations were mentioned as specific examples of how Florida has become a favorite destination for Republicans.


  2%|▏         | 2/100 [02:31<2:03:31, 75.63s/it]

(Palm Beach, Florida, 1, Mar-a-Lago, Donald Trump, 2, CPAC, Conservative Political Action Conference, Hyatt Regency, Orlando, Florida, 3, RNC, Florida, Florida, Republicans)
Time taken: 00:02:31


Llama.generate: prefix-match hit

llama_print_timings:        load time =   96255.41 ms
llama_print_timings:      sample time =      18.36 ms /    17 runs   (    1.08 ms per token,   926.13 tokens per second)
llama_print_timings: prompt eval time =   98690.43 ms /  1146 tokens (   86.12 ms per token,    11.61 tokens per second)
llama_print_timings:        eval time =    3002.45 ms /    16 runs   (  187.65 ms per token,     5.33 tokens per second)
llama_print_timings:       total time =  101749.32 ms /  1162 tokens


...

(Boston, Massachusetts)
Time taken: 00:01:42


Llama.generate: prefix-match hit

llama_print_timings:        load time =   96255.41 ms
llama_print_timings:      sample time =      45.37 ms /    36 runs   (    1.26 ms per token,   793.42 tokens per second)
llama_print_timings: prompt eval time =   93728.56 ms /  1105 tokens (   84.82 ms per token,    11.79 tokens per second)
llama_print_timings:        eval time =    7184.54 ms /    35 runs   (  205.27 ms per token,     4.87 tokens per second)
llama_print_timings:       total time =  101086.01 ms /  1140 tokens




The most exact real location mentioned in the article is:

WASHINGTON (AP)

This refers to the city of Washington, which is located in the state of Maryland, USA.


100%|██████████| 100/100 [1:55:46<00:00, 69.47s/it]

(WASHINGTON, AP, Washington, Maryland, USA)
Time taken: 00:01:46
Total time taken: 01:55:47


In [46]:
df.head(10)

,_id,hl1,body,Explicit_Pass,NER_Pass,LLM_2_Pass,LLM_3_1_Pass
7702,0000017f-263d-d19f-a3ff-ef3d8a100001,Florida has become the GOP favorite destinatio...,Florida is big draw for snow birds from around...,None,Mar Lago,Mar-a-Lago,Mar-a-Lago
2936,00000178-56fd-da59-a37b-d7fd12e40001,Year Into Pandemic Veterans Halls Barely Hangi...,NEW BEDFORD Mass. (AP) Paul Guilbeault knew th...,None,None,VFW,None
6426,0000017d-203f-d269-a3fd-be3ffda40000,dead hundreds injured after storms rouse scorp...,Three people are dead and hundreds are injured...,None,None,Saint Louis Zoo,Al Ahram
6693,0000017d-9619-d0ab-a17d-deffaa9e0001,Boston Ballet The Gift,Enjoy special preview of Boston Ballet The Gif...,None,None,the Boston Opera House,Boston Ballet
4389,0000017a-26b7-dc5c-a77a-2ef7a0a40001,Driver Rams Cyclists In Arizona Race Criticall...,SHOW LOW Ariz. (AP) driver in pickup truck plo...,None,U.S. 60,Main Street,U.S. 60
2581,00000177-f7bf-d081-ad7f-ffbf11190001,Drug Overdose Deaths Surge Among Black America...,When Latoya Jenkins talks about her mom she li...,None,None,the Centers for Disease Control and Prevention,University of Pennsylvania
2344,00000177-ba85-d244-a57f-fbfd3e720001,The Outlaw And The Lawman Remembrance Of Class...,It was the classic tale of the outlaw and the ...,None,None,Winter Hill,The Winter Hill gang
1776,00000177-2ad9-d79b-abff-abffb43c0001,LISTEN American Democracy Buy Or Sell,After the 2020 presidential election and the h...,None,None,GBH News,None
10526,00000183-af56-d227-a9b7-bfd7c9ce0001,Boston Latino business owners still lacking ac...,Dr. Rosa Calca of Boston owns two businesses b...,None,Seaport,Seaport,None
2748,00000178-235f-da59-a37b-a7ffd5f40001,Campbell And Wu Criticize Walsh Handling Of Su...,Two candidates to replace Mayor Marty Walsh ha...,None,None,the Boston Police Department,None


In [54]:
df

,_id,hl1,body,Explicit_Pass,NER_Pass,LLM_2_Pass,LLM_3_1_Pass
7702,0000017f-263d-d19f-a3ff-ef3d8a100001,Florida has become the GOP favorite destinatio...,Florida is big draw for snow birds from around...,None,Mar Lago,Mar-a-Lago,Mar-a-Lago
2936,00000178-56fd-da59-a37b-d7fd12e40001,Year Into Pandemic Veterans Halls Barely Hangi...,NEW BEDFORD Mass. (AP) Paul Guilbeault knew th...,None,None,VFW,None
6426,0000017d-203f-d269-a3fd-be3ffda40000,dead hundreds injured after storms rouse scorp...,Three people are dead and hundreds are injured...,None,None,Saint Louis Zoo,Al Ahram
6693,0000017d-9619-d0ab-a17d-deffaa9e0001,Boston Ballet The Gift,Enjoy special preview of Boston Ballet The Gif...,None,None,the Boston Opera House,Boston Ballet
4389,0000017a-26b7-dc5c-a77a-2ef7a0a40001,Driver Rams Cyclists In Arizona Race Criticall...,SHOW LOW Ariz. (AP) driver in pickup truck plo...,None,U.S. 60,Main Street,U.S. 60
...,...,...,...,...,...,...,...
12243,00000186-5ae3-d0c8-adc7-5ef35ee20001,50 years later one museum remembers American v...,Jared Bowen on Boston Public Radio Feb 15 2023...,None,Hanoi Hilton,the National Museum of Women in the Arts,Hoa Lo Prison
170,00000175-9d42-d5c8-a775-bf4efe400001,Biden Overtakes Trump In Georgia Vote Count,Democrat Joe Biden is now leading President Do...,None,None,AP,Associated Press
1582,00000176-fe28-d79b-abfe-ffbe8aa70001,The Framers Understood That Such President Had...,For the second time President Donald Trump has...,None,Capitol,The White House,Congress
11069,00000184-8cd5-dce9-a5b6-8dff4e7b0001,Boston LGBTQ documentary film festival showcas...,This weekend Wicked Queer Boston LGBTQ Film Fe...,None,the Brattle Theatre,the Museum of Fine Arts,None


In [47]:
time_df

,NER,LLM2,LLM3.1
0,02:14:16,03:34:59,01:55:47


In [48]:
print(df['Explicit_Pass'].value_counts().sum())
df['Explicit_Pass'].value_counts()

0


Series([], Name: count, dtype: int64)

In [49]:
print(df['NER_Pass'].value_counts().sum())
df['NER_Pass'].value_counts()

36


NER_Pass
Capitol                                      6
Mar Lago                                     1
Connell                                      1
Hanoi Hilton                                 1
Lynn Common                                  1
Hotel Portofino                              1
Everett High                                 1
Madison Park                                 1
City Winery                                  1
State House                                  1
Peabody Essex Museum                         1
Burbank Hospital                             1
San Francisco International Airport          1
the Brooke House College Football Academy    1
Robb Elementary School                       1
MBTA                                         1
Suez Canal                                   1
U.S. 60                                      1
Oak Grove                                    1
Paris Charles de Gaulle Airport              1
ICA                                          1
the 

In [50]:
print(df['LLM_2_Pass'].value_counts().sum())
df['LLM_2_Pass'].value_counts()

92


LLM_2_Pass
Harvard University                                4
Boston Public Radio                               2
Capitol                                           2
The White House                                   2
MBTA                                              2
                                                 ..
the U.S. Supreme Court                            1
WGBH                                              1
the GBH Calderwood Studio                         1
Amrheins                                          1
The Centers for Disease Control and Prevention    1
Name: count, Length: 84, dtype: int64

In [51]:
print(df['LLM_3_1_Pass'].value_counts().sum())
df['LLM_3_1_Pass'].value_counts()

63


LLM_3_1_Pass
The U.S. Capitol                          2
Harvard University                        2
Mar-a-Lago                                1
Trader Joe                                1
Environmental Protection Agency           1
                                         ..
Oleana                                    1
Institute of Contemporary Art             1
Transportation Security Administration    1
East Boston Neighborhood Health Center    1
AP                                        1
Name: count, Length: 61, dtype: int64

In [52]:
len(df)

100

In [53]:
df.to_csv(f"./results/benchmarking_{sample_count}_samples_{trial}_trial.csv")
time_df.to_csv(f"./results/benchmarking_times_{sample_count}_samples_{trial}_trial.csv")

In [73]:
print(df.count())
print(df.dropna(subset=['NER_Pass'])["LLM_2_Pass"].notnull().sum())
print(df.dropna(subset=['NER_Pass'])["LLM_3_1_Pass"].notnull().sum())

print(df.dropna(subset=['LLM_2_Pass'])["LLM_3_1_Pass"].notnull().sum())

_id              100
hl1              100
body             100
Explicit_Pass      0
NER_Pass          36
LLM_2_Pass        92
LLM_3_1_Pass      63
different        100
dtype: int64
35
28
61


In [91]:
df2 = df.copy()
df2.drop(columns=['hl1', 'body', 'Explicit_Pass', 'different'], inplace=True)

df3 = df2.dropna(subset=['NER_Pass'])
print(df3.count())
df3

_id             36
NER_Pass        36
LLM_2_Pass      35
LLM_3_1_Pass    28
dtype: int64


,_id,NER_Pass,LLM_2_Pass,LLM_3_1_Pass
7702,0000017f-263d-d19f-a3ff-ef3d8a100001,Mar Lago,Mar-a-Lago,Mar-a-Lago
4389,0000017a-26b7-dc5c-a77a-2ef7a0a40001,U.S. 60,Main Street,U.S. 60
10526,00000183-af56-d227-a9b7-bfd7c9ce0001,Seaport,Seaport,None
6309,0000017c-ecc7-dfb8-af7d-ede7af640001,Encore Boston Harbor,Encore Boston Harbor,MGM Springfield
10530,00000183-b1e3-d4ba-a9a7-bdfb61830002,Capitol,Twitter,Twitter
2659,00000178-0fa2-d1f8-a57e-4fa2a3da0001,Boston Marathon,Marathon,Michigan State University
421,00000175-d71b-dd9a-a37d-ffdff1d20001,the State House,the State House,The State House
1447,00000176-e203-d1d4-a57f-ee5b6f1b0001,Capitol,Capitol,The U.S. Capitol
2965,00000178-5c33-d0b9-abfc-fe338b1d0001,Faneuil Hall,Faneuil Hall,None
10553,00000183-b4a5-d0d0-adfb-fda53e7d0001,Logan Airport,Amrheins,None


In [95]:
df4 = df2.dropna(subset=['LLM_2_Pass'])
print(df4.count())
df4

_id             63
NER_Pass        28
LLM_2_Pass      61
LLM_3_1_Pass    63
dtype: int64


,_id,NER_Pass,LLM_2_Pass,LLM_3_1_Pass
7702,0000017f-263d-d19f-a3ff-ef3d8a100001,Mar Lago,Mar-a-Lago,Mar-a-Lago
6426,0000017d-203f-d269-a3fd-be3ffda40000,None,Saint Louis Zoo,Al Ahram
6693,0000017d-9619-d0ab-a17d-deffaa9e0001,None,the Boston Opera House,Boston Ballet
4389,0000017a-26b7-dc5c-a77a-2ef7a0a40001,U.S. 60,Main Street,U.S. 60
2581,00000177-f7bf-d081-ad7f-ffbf11190001,None,the Centers for Disease Control and Prevention,University of Pennsylvania
...,...,...,...,...
9857,00000182-7f38-db6a-afee-7fb888000001,Lynn Common,the Lynn Common,Lynn Common
12243,00000186-5ae3-d0c8-adc7-5ef35ee20001,Hanoi Hilton,the National Museum of Women in the Arts,Hoa Lo Prison
170,00000175-9d42-d5c8-a775-bf4efe400001,None,AP,Associated Press
1582,00000176-fe28-d79b-abfe-ffbe8aa70001,Capitol,The White House,Congress
